Выгрузим данные и лучшую нашу модель

In [1]:
import joblib
import pandas as pd
from sklearn.metrics import mean_absolute_error, r2_score


X_test = pd.read_parquet('../data/future_testing/X_test_final.parquet')
y_test = pd.read_parquet('../data/future_testing/y_test_final.parquet')

model = joblib.load('../models/lightgbm_v1.pkl')

print("Входные признаки (включая als_score):")
display(X_test.head())
print(f"Размер таблицы X_test: {X_test.shape}")

print("\nРеальные ответы (target):")
display(y_test.head())
print(f"Размер таблицы y_test: {y_test.shape}")

Входные признаки (включая als_score):


,place,platform,agent,like,dislike,share,bookmark,click_on_author,open_comments,duration,...,engagement_score,author_avg_target,author_avg_engagement,author_popularity,user_avg_target,user_avg_engagement,user_activity,video_duration_type,als_score,embedding_similarity_score
index,,,,,,,,,,,,,,,,,,,,,
10117391,0,0,0,0,0,0,0,0,0,52,...,0,0.513505,0.091533,3059,0.651635,0.000000,930,3,0.077845,0.814611
26635015,2,0,0,0,0,0,0,0,0,45,...,0,0.542583,0.022603,3097,0.291384,0.005398,741,3,0.067293,0.529881
37339584,0,0,0,0,0,0,0,0,0,73,...,0,0.562741,0.004223,25340,0.294076,0.000000,57,3,0.003684,0.471301
22954482,0,0,0,0,0,0,0,0,0,60,...,0,0.462244,0.022472,26299,0.727800,0.000885,3388,3,0.027754,0.284203
30333544,2,0,0,0,0,0,0,0,0,16,...,0,0.601148,0.026133,65473,0.773216,0.000000,112,2,0.034177,0.715246


Размер таблицы X_test: (1000000, 25)

Реальные ответы (target):


,target
index,
10117391,1.000000
26635015,0.111111
37339584,0.232877
22954482,0.016667
30333544,0.437500


Размер таблицы y_test: (1000000, 1)


In [2]:
import numpy as np
from sklearn.metrics import mean_absolute_error, r2_score

y_pred = model.predict(X_test)

y_pred = np.clip(y_pred, 0, 1)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Средняя ошибка (Watch Ratio MAE): {mae:.4f}")
print(f"Коэффициент детерминации (R2): {r2:.4f}")

Средняя ошибка (Watch Ratio MAE): 0.2749
Коэффициент детерминации (R2): 0.1528


In [3]:
comparison_df = pd.DataFrame({
    'Real_Ratio': y_test.values.flatten(),
    'Predicted_Ratio': y_pred,
    'Duration_Sec': X_test['duration'].values
})

# Считаем просмотр в секундах
comparison_df['Real_Seconds'] = comparison_df['Real_Ratio'] * comparison_df['Duration_Sec']
comparison_df['Predicted_Seconds'] = comparison_df['Predicted_Ratio'] * comparison_df['Duration_Sec']

# Ошибка в секундах (модуль разности)
comparison_df['Error_Seconds'] = (comparison_df['Real_Seconds'] - comparison_df['Predicted_Seconds']).abs()

print("Сравнение реального времени просмотра и предсказанного (в секундах):")
display(comparison_df[['Real_Seconds', 'Predicted_Seconds', 'Error_Seconds', 'Duration_Sec']].head(100))

Сравнение реального времени просмотра и предсказанного (в секундах):


,Real_Seconds,Predicted_Seconds,Error_Seconds,Duration_Sec
0,52.0,48.783107,3.216893,52
1,5.0,4.784540,0.215460,45
2,17.0,5.987181,11.012819,73
3,1.0,47.368662,46.368662,60
4,7.0,13.263761,6.263761,16
...,...,...,...,...
95,2.0,4.856775,2.856775,81
96,50.0,50.999995,0.999995,51
97,5.0,35.831265,30.831265,36
98,12.0,11.695944,0.304056,12


In [4]:
mae_seconds = comparison_df['Error_Seconds'].mean()
print(f"Средняя абсолютная ошибка: {mae_seconds:.2f} секунд")

median_error_sec = comparison_df['Error_Seconds'].median()
print(f"Медианная абсолютная ошибка: {median_error_sec:.2f} секунд")

Средняя абсолютная ошибка: 13.70 секунд
Медианная абсолютная ошибка: 6.54 секунд
